In [1]:
# ==========================================
# CÓDIGO 1: ANÁLISE DE ESTABILIDADE (FLIP-RATE)
# BASE: STUDENT PERFORMANCE (OpenML data_id=42352)
# ==========================================

import os
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from scipy.sparse.csgraph import connected_components

from deap import base, creator, tools
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=SyntaxWarning)

# =========================================================
# Bibliotecas locais
# =========================================================
import utils
import phyil

# =========================================================
# Configurações globais
# =========================================================
out_dir = os.path.join(os.getcwd(), "student_performance")
os.makedirs(out_dir, exist_ok=True)

QUANTIS_LISTA = [2, 4, 8, 16]
SIGMAS = np.arange(0.01, 0.31, 0.01)
NUM_PERT = 30
SEED_GLOBAL = 42

# =========================================================
# Funções de suporte
# =========================================================

def apagar_todas_imagens_png_da_pasta(diretorio):
    if not os.path.exists(diretorio):
        return

    arquivos_removidos = []

    for nome_arquivo in os.listdir(diretorio):
        caminho = os.path.join(diretorio, nome_arquivo)

        if os.path.isfile(caminho) and nome_arquivo.lower().endswith(".png"):
            try:
                os.remove(caminho)
                arquivos_removidos.append(nome_arquivo)
            except Exception as e:
                print(f"[Aviso] Não foi possível apagar '{nome_arquivo}'. Erro: {e}")

    print(f"-> Limpeza inicial: {len(arquivos_removidos)} imagens PNG antigas apagadas.")


def carregar_student_performance():
    """
    Carrega a base Student Performance do OpenML.

    Ground Truth:
        G3 >= 10 -> 1 (sucesso acadêmico)
        G3 < 10  -> 0 (insucesso acadêmico)

    G3 é removida das variáveis preditoras para impedir leakage.
    """
    print("-> Baixando base Student Performance do OpenML (data_id=42352)...")

    dados = fetch_openml(data_id=42352, as_frame=True)
    df = dados.frame.copy()

    colunas_necessarias = [
        "failures",
        "absences",
        "studytime",
        "health",
        "freetime",
        "G3",
    ]

    colunas_ausentes = [c for c in colunas_necessarias if c not in df.columns]
    if colunas_ausentes:
        raise ValueError(
            f"As colunas esperadas não foram encontradas: {colunas_ausentes}. "
            f"Colunas disponíveis: {df.columns.tolist()}"
        )

    for coluna in colunas_necessarias:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    df = df.dropna(subset=colunas_necessarias).reset_index(drop=True)

    # Alvo real usado apenas para avaliação e baseline-oráculo
    y_true = (df["G3"] >= 10).astype(int).to_numpy()

    # Remove G3 para evitar vazamento do ground truth nas features.
    X_features = df.drop(columns=["G1", "G2", "G3"]).copy()

    colunas_categoricas = X_features.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    if colunas_categoricas:
        X_features = pd.get_dummies(
            X_features,
            columns=colunas_categoricas,
            drop_first=True,
            dtype=float,
        )

    for coluna in X_features.columns:
        X_features[coluna] = pd.to_numeric(X_features[coluna], errors="coerce")

    X_features = X_features.fillna(0)

    print(f"-> Base carregada: {len(df)} estudantes.")
    print(
        f"-> Classe 0 (G3 < 10): {(y_true == 0).sum()} | "
        f"Classe 1 (G3 >= 10): {(y_true == 1).sum()}"
    )

    return df, X_features, y_true


def obter_objetivos_student_performance(df_base):
    """
    Objetivos definidos no artigo:

    failures  -> minimizar
    absences  -> minimizar
    studytime -> maximizar
    health    -> maximizar
    freetime  -> minimizar
    """
    failures = df_base["failures"].astype(float).to_numpy()
    absences = df_base["absences"].astype(float).to_numpy()
    studytime = df_base["studytime"].astype(float).to_numpy()
    health = df_base["health"].astype(float).to_numpy()
    freetime = df_base["freetime"].astype(float).to_numpy()

    return np.column_stack(
        (failures, absences, studytime, health, freetime)
    )


def sanitizar_matriz_distancias(dist_matrix):
    dist_matrix = np.asarray(dist_matrix, dtype=float)
    dist_matrix = 0.5 * (dist_matrix + dist_matrix.T)
    np.fill_diagonal(dist_matrix, 0.0)
    return dist_matrix


def obter_matriz_distancia_geodesica(X_subset, num_vizinhos):
    try:
        resultado_dist = utils.matrizDistancia(
            X_subset,
            tipo="geodesica",
            num_vizinhos=num_vizinhos,
        )
    except Exception:
        resultado_dist = utils.matrizDistancia(X_subset, tipo="geodesica")

    dist_matrix = (
        resultado_dist[-1]
        if isinstance(resultado_dist, tuple)
        else resultado_dist
    )

    if hasattr(dist_matrix, "toarray"):
        dist_matrix = dist_matrix.toarray()

    return sanitizar_matriz_distancias(dist_matrix)


def checar_conectividade_knn(X_subset, n_neighbors=6):
    n_neighbors = min(n_neighbors, len(X_subset) - 1)

    nbrs = NearestNeighbors(
        n_neighbors=n_neighbors,
        algorithm="auto",
        n_jobs=-1,
    ).fit(X_subset)

    knnG = kneighbors_graph(
        nbrs,
        n_neighbors=n_neighbors,
        mode="distance",
        n_jobs=-1,
    )

    knnG = knnG.maximum(knnG.T)
    n_comp, _ = connected_components(knnG, directed=False)

    return n_comp == 1


def encontrar_k_conexo(X_subset, k_inicial=6):
    n_amostras = len(X_subset)

    for k in range(k_inicial, n_amostras):
        if checar_conectividade_knn(X_subset, n_neighbors=k):
            return k

    raise ValueError("Grafo desconexo.")


def run_phyil_core(X_subset):
    k_escolhido = encontrar_k_conexo(X_subset)

    dist_matrix = obter_matriz_distancia_geodesica(
        X_subset,
        num_vizinhos=k_escolhido,
    )

    dist_df_norm = phyil.normalize_df_0_1(pd.DataFrame(dist_matrix))
    dist_matrix_norm = sanitizar_matriz_distancias(dist_df_norm.values)

    grupos, _, tree = utils.damicore(
        X=pd.DataFrame(dist_matrix_norm),
        y=None,
        distancias=dist_matrix_norm,
        n_neighbors=k_escolhido,
        algo_deteccao="fastgreedy",
    )

    selected = utils.chooseSamplesToLabel(
        grupos=grupos,
        X=X_subset,
        tree=tree,
        newick_str="nj_tree.newick",
        type="NodeInFilogramCenter",
    )

    return np.array(selected)


def run_nds_deap(X_obj, weights):
    if hasattr(creator, "FitMultiMix"):
        del creator.FitMultiMix

    if hasattr(creator, "IndMultiMix"):
        del creator.IndMultiMix

    creator.create("FitMultiMix", base.Fitness, weights=weights)
    creator.create("IndMultiMix", list, fitness=creator.FitMultiMix)

    pop = [creator.IndMultiMix([i]) for i in range(len(X_obj))]

    for i, ind in enumerate(pop):
        ind.fitness.values = tuple(X_obj[i])

    fronts = tools.sortNondominated(
        pop,
        k=len(pop),
        first_front_only=False,
    )

    return np.array([ind[0] for front in fronts for ind in front])


def perturbar_grafo_local(W, seed):
    rng = np.random.default_rng(seed)

    n_rows, n_cols = W.shape
    mask_off = ~np.eye(n_rows, dtype=bool)
    std_off = np.std(W[mask_off])

    pert_idx = rng.choice(
        n_rows * n_cols,
        size=min(100, n_rows * n_cols),
        replace=False,
    )

    rows, cols = np.unravel_index(pert_idx, W.shape)
    noise = rng.normal(0, 0.5 * std_off, size=len(pert_idx))

    W_pert = W.copy()
    W_pert[rows, cols] = np.clip(W[rows, cols] + noise, 0, 1)
    W_pert = 0.5 * (W_pert + W_pert.T)

    np.fill_diagonal(W_pert, 1.0)

    return W_pert


def run_llgc_precomputed(W, labeled_indices, labels):
    y = np.full(W.shape[0], -1, dtype=int)
    y[labeled_indices] = np.asarray(labels, dtype=int)

    def custom_kernel(X, Y=None):
        return W

    model = LabelSpreading(
        kernel=custom_kernel,
        alpha=0.2,
        max_iter=50,
    )

    model.fit(np.zeros((W.shape[0], 1)), y)

    return model.transduction_.astype(int)


# =========================================================
# Pipeline principal
# =========================================================

apagar_todas_imagens_png_da_pasta(out_dir)
np.random.seed(SEED_GLOBAL)

df_base, X_df, y_true = carregar_student_performance()
X = StandardScaler().fit_transform(X_df.values)

X_obj = obter_objetivos_student_performance(df_base)

print("-> Preparando matrizes globais...")

D_norm_euc = sanitizar_matriz_distancias(
    phyil.normalize_df_0_1(
        pd.DataFrame(euclidean_distances(X))
    ).values
)

D_norm_geo = sanitizar_matriz_distancias(
    phyil.normalize_df_0_1(
        pd.DataFrame(obter_matriz_distancia_geodesica(X, 6))
    ).values
)

print("-> Processando NDS...")

# failures (-), absences (-), studytime (+), health (+), freetime (-)
ranked_indices = run_nds_deap(
    X_obj,
    weights=(-1.0, -1.0, 1.0, 1.0, -1.0),
)

resultados = []
rng = np.random.default_rng(SEED_GLOBAL)

for q in QUANTIS_LISTA:
    print(f"\n-> Processando Flip-Rate para {q} quantis...")

    quantis = np.array_split(ranked_indices, q)

    idx_inf = run_phyil_core(X[quantis[0]])
    idx_sup = run_phyil_core(X[quantis[-1]])

    seeds_hermes = np.concatenate(
        [quantis[0][idx_inf], quantis[-1][idx_sup]]
    )

    labels_hermes = np.concatenate(
        [np.zeros(len(idx_inf)), np.ones(len(idx_sup))]
    )

    seeds_b1 = rng.choice(
        np.arange(len(X)),
        size=len(seeds_hermes),
        replace=False,
    )

    labels_b1 = y_true[seeds_b1].astype(int)

    for sigma in tqdm(SIGMAS, desc="Varredura Sigma"):
        sig = np.round(float(sigma), 3)

        W_ori_h = np.exp(
            -np.square(D_norm_geo) / (2 * sig ** 2)
        )
        np.fill_diagonal(W_ori_h, 1.0)

        W_ori_b = np.exp(
            -np.square(D_norm_euc) / (2 * sig ** 2)
        )
        np.fill_diagonal(W_ori_b, 1.0)

        p_ref_h = run_llgc_precomputed(
            W_ori_h,
            seeds_hermes,
            labels_hermes,
        )

        p_ref_b = run_llgc_precomputed(
            W_ori_b,
            seeds_b1,
            labels_b1,
        )

        for pert in range(1, NUM_PERT + 1):
            p_h = run_llgc_precomputed(
                perturbar_grafo_local(
                    W_ori_h,
                    int(sig * 1000) + pert * 10 + 999,
                ),
                seeds_hermes,
                labels_hermes,
            )

            p_b = run_llgc_precomputed(
                perturbar_grafo_local(
                    W_ori_b,
                    int(sig * 1000) + pert * 10 + 111,
                ),
                seeds_b1,
                labels_b1,
            )

            resultados.append(
                {
                    "q": q,
                    "sigma": sig,
                    "perturbacao": pert,
                    "modelo": "HERMES",
                    "flip_rate": np.mean(p_ref_h != p_h),
                }
            )

            resultados.append(
                {
                    "q": q,
                    "sigma": sig,
                    "perturbacao": pert,
                    "modelo": "Baseline Aleatório",
                    "flip_rate": np.mean(p_ref_b != p_b),
                }
            )

print("\n-> Salvando resultados...")

df_res = pd.DataFrame(resultados)

df_res.to_csv(
    os.path.join(out_dir, "dados_flip_rate_brutos.csv"),
    index=False,
    sep=";",
    decimal=",",
)

df_grouped = (
    df_res.groupby(["q", "sigma", "modelo"])
    .agg(
        mean_flip=("flip_rate", "mean"),
        std_flip=("flip_rate", "std"),
        n=("flip_rate", "count"),
    )
    .reset_index()
)

z = 1.96
se = df_grouped["std_flip"] / np.sqrt(df_grouped["n"])

df_grouped["ci_low"] = df_grouped["mean_flip"] - z * se
df_grouped["ci_high"] = df_grouped["mean_flip"] + z * se

df_grouped.to_csv(
    os.path.join(out_dir, "dados_flip_rate.csv"),
    index=False,
    sep=";",
    decimal=",",
)

print(f"-> Concluído! Resultados salvos em: {out_dir}")

-> Limpeza inicial: 40 imagens PNG antigas apagadas.
-> Baixando base Student Performance do OpenML (data_id=42352)...
-> Base carregada: 395 estudantes.
-> Classe 0 (G3 < 10): 130 | Classe 1 (G3 >= 10): 265
-> Preparando matrizes globais...
-> Processando NDS...

-> Processando Flip-Rate para 2 quantis...


Varredura Sigma: 100%|██████████| 30/30 [00:09<00:00,  3.29it/s]



-> Processando Flip-Rate para 4 quantis...


Varredura Sigma: 100%|██████████| 30/30 [00:07<00:00,  3.99it/s]



-> Processando Flip-Rate para 8 quantis...


Varredura Sigma: 100%|██████████| 30/30 [00:07<00:00,  3.94it/s]



-> Processando Flip-Rate para 16 quantis...


Varredura Sigma: 100%|██████████| 30/30 [00:07<00:00,  4.17it/s]


-> Salvando resultados...
-> Concluído! Resultados salvos em: D:\pycharm_projects\HERMES_diabetes_and_student\student_performance


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score


def evaluate_metrics(y_true_slice, pred_slice):
    y_true_slice = np.asarray(y_true_slice, dtype=int)
    pred_slice = np.asarray(pred_slice, dtype=int)

    return {
        "accuracy": accuracy_score(y_true_slice, pred_slice),
        "f1": f1_score(
            y_true_slice,
            pred_slice,
            average="binary",
            pos_label=1,
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_true_slice,
            pred_slice,
            average="macro",
            zero_division=0,
        ),
    }


def align_unsupervised_metrics(y_true_slice, pred_slice):
    pred_direct = np.asarray(pred_slice, dtype=int)
    pred_flipped = 1 - pred_direct

    metrics_direct = evaluate_metrics(y_true_slice, pred_direct)
    metrics_flipped = evaluate_metrics(y_true_slice, pred_flipped)

    if metrics_direct["accuracy"] >= metrics_flipped["accuracy"]:
        metrics_direct["orientation"] = "direct"
        return metrics_direct

    metrics_flipped["orientation"] = "flipped"
    return metrics_flipped


def append_result(
    results,
    q,
    model,
    family,
    round_number,
    metrics_10,
    metrics_25,
    metrics_total,
):
    results.append(
        {
            "q": q,
            "modelo": model,
            "familia": family,
            "rodada": round_number,
            "acc_10": metrics_10["accuracy"],
            "f1_10": metrics_10["f1"],
            "f1_macro_10": metrics_10["f1_macro"],
            "orientacao_10": metrics_10.get("orientation", "direct"),
            "acc_25": metrics_25["accuracy"],
            "f1_25": metrics_25["f1"],
            "f1_macro_25": metrics_25["f1_macro"],
            "orientacao_25": metrics_25.get("orientation", "direct"),
            "acc_total": metrics_total["accuracy"],
            "f1_total": metrics_total["f1"],
            "f1_macro_total": metrics_total["f1_macro"],
            "orientacao_total": metrics_total.get("orientation", "direct"),
        }
    )


OUT_DIR = Path.cwd() / "student_performance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# SIGMAS_ESCOLHIDOS_HERMES = {
#     2: [0.01, 0.02, 0.03, 0.04],
#     4: [0.01, 0.02, 0.03, 0.04],
#     8: [0.01, 0.02, 0.03, 0.04],
#     16: [0.01, 0.02, 0.03, 0.04],
# }
#
# SIGMAS_ESCOLHIDOS_B = {
#     2: [0.02, 0.03, 0.04],
#     4: [0.02, 0.03, 0.04],
#     8: [0.02, 0.03, 0.04],
#     16: [0.02, 0.03, 0.04],
# }

# ----------------------------------------------------------------
# 1. SIGMAS ESCOLHIDOS PARA O HERMES
# ----------------------------------------------------------------

SIGMAS_ESCOLHIDOS_HERMES = {
    2: [0.03],
    4: [0.03],
    8: [0.03, 0.04],
    16: [0.04],
}


# ----------------------------------------------------------------
# 2. SIGMAS ESCOLHIDOS PARA O BASELINE 1
# ----------------------------------------------------------------

SIGMAS_ESCOLHIDOS_B = {
    2: [0.07, 0.08],
    4: [0.07],
    8: [0.08],
    16: [0.07],
}

NUM_RODADAS_ALEATORIAS = 30
resultados_perf = []

k10 = max(1, int(len(ranked_indices) * 0.10))
idx_conc_10 = np.concatenate(
    [ranked_indices[:k10], ranked_indices[-k10:]]
)

k25 = max(1, int(len(ranked_indices) * 0.25))
idx_conc_25 = np.concatenate(
    [ranked_indices[:k25], ranked_indices[-k25:]]
)

for q in tqdm(QUANTIS_LISTA, desc="Quantis"):
    quantis = np.array_split(ranked_indices, q)

    sigmas_h_q = SIGMAS_ESCOLHIDOS_HERMES[q]
    sigmas_b_q = SIGMAS_ESCOLHIDOS_B[q]

    idx_inf = run_phyil_core(X[quantis[0]])
    idx_sup = run_phyil_core(X[quantis[-1]])

    seeds_hermes = np.concatenate(
        [quantis[0][idx_inf], quantis[-1][idx_sup]]
    )

    labels_hermes = np.concatenate(
        [
            np.zeros(len(idx_inf), dtype=int),
            np.ones(len(idx_sup), dtype=int),
        ]
    )

    n_seeds = len(seeds_hermes)

    for sig in sigmas_h_q:
        W_geo = np.exp(-np.square(D_norm_geo) / (2 * sig ** 2))
        np.fill_diagonal(W_geo, 1.0)

        p_h = run_llgc_precomputed(
            W_geo,
            seeds_hermes,
            labels_hermes,
        )

        metrics_10 = align_unsupervised_metrics(
            y_true[idx_conc_10],
            p_h[idx_conc_10],
        )

        metrics_25 = align_unsupervised_metrics(
            y_true[idx_conc_25],
            p_h[idx_conc_25],
        )

        metrics_total = align_unsupervised_metrics(y_true, p_h)

        append_result(
            resultados_perf,
            q,
            f"HERMES ($\\sigma$={sig:.2f})",
            "HERMES",
            1,
            metrics_10,
            metrics_25,
            metrics_total,
        )

    for rodada in range(1, NUM_RODADAS_ALEATORIAS + 1):
        rng = np.random.default_rng(SEED_GLOBAL + rodada)

        seeds_random = rng.choice(
            np.arange(len(X)),
            size=n_seeds,
            replace=False,
        )

        labels_random = y_true[seeds_random].astype(int)

        for sig in sigmas_b_q:
            W_b = np.exp(-np.square(D_norm_euc) / (2 * sig ** 2))
            np.fill_diagonal(W_b, 1.0)

            p_b = run_llgc_precomputed(
                W_b,
                seeds_random,
                labels_random,
            )

            metrics_10_base = evaluate_metrics(
                y_true[idx_conc_10],
                p_b[idx_conc_10],
            )

            metrics_25_base = evaluate_metrics(
                y_true[idx_conc_25],
                p_b[idx_conc_25],
            )

            metrics_total_base = evaluate_metrics(y_true, p_b)

            append_result(
                resultados_perf,
                q,
                f"Baseline Aleatório ($\\sigma$={sig:.2f})",
                "Baseline Aleatório",
                rodada,
                metrics_10_base,
                metrics_25_base,
                metrics_total_base,
            )

df_res = pd.DataFrame(resultados_perf)

ordem_colunas = [
    "q",
    "modelo",
    "familia",
    "rodada",
    "acc_10",
    "f1_10",
    "f1_macro_10",
    "orientacao_10",
    "acc_25",
    "f1_25",
    "f1_macro_25",
    "orientacao_25",
    "acc_total",
    "f1_total",
    "f1_macro_total",
    "orientacao_total",
]

df_res = df_res[ordem_colunas]

arquivo_saida = OUT_DIR / "dados_sigmas_escolhidos.csv"

df_res.to_csv(
    arquivo_saida,
    index=False,
    sep=";",
    decimal=",",
)

print(f"\n-> Experimento concluído. Resultados salvos em: {arquivo_saida}")

Quantis: 100%|██████████| 4/4 [00:10<00:00,  2.66s/it]


-> Experimento concluído. Resultados salvos em: D:\pycharm_projects\HERMES_diabetes_and_student\student_performance\dados_sigmas_escolhidos.csv


In [3]:
# =========================================================
# GERADOR DE GRÁFICOS
# Flip-Rate: HERMES vs Baseline Aleatório
# Performance: HERMES vs Baseline Aleatório
# =========================================================

import os
import pandas as pd

# =========================================================
# 1. CONFIGURAÇÃO DO DIRETÓRIO E CAMINHOS
# =========================================================

out_dir = os.path.join(os.getcwd(), "student_performance")

csv_flip_rate = os.path.join(out_dir, "dados_flip_rate.csv")
csv_performance = os.path.join(out_dir, "dados_sigmas_escolhidos.csv")

if not os.path.exists(csv_flip_rate) or not os.path.exists(csv_performance):
    print(
        "⚠️ Aviso: não encontrei os ficheiros CSV. "
        "Execute primeiro os experimentos de flip-rate e desempenho."
    )
else:
    print(f"-> Lendo dados da pasta: {out_dir}")

    # Configuração global de estilo para artigos científicos
    sns.set_theme(
        style="whitegrid",
        context="paper",
        font_scale=1.2
    )

    # =========================================================
    # 2. GRÁFICOS DE ESTABILIDADE: FLIP-RATE
    # =========================================================

    print("-> Gerando gráficos de Flip-Rate...")

    df_flip = pd.read_csv(csv_flip_rate, sep=";", decimal=",")
    quantis_flip = sorted(df_flip["q"].unique())

    # Paleta de cores e marcadores atualizada com o novo baseline
    cores_flip = {
        "HERMES": "#1f77b4",
        "Baseline Aleatório": "#d62728"
    }

    marcadores_flip = {
        "HERMES": "o",
        "Baseline Aleatório": "s"
    }

    for q in quantis_flip:
        df_q = df_flip[df_flip["q"] == q].copy().sort_values("sigma")

        if df_q.empty:
            continue

        plt.figure(figsize=(10, 6))

        for modelo in ["HERMES", "Baseline Aleatório"]:
            sub = df_q[df_q["modelo"] == modelo].copy().sort_values("sigma")

            if sub.empty:
                continue

            x = sub["sigma"].to_numpy()
            y = sub["mean_flip"].to_numpy() * 100.0
            y_low = sub["ci_low"].to_numpy() * 100.0
            y_high = sub["ci_high"].to_numpy() * 100.0

            # Plotagem da linha principal (média)
            plt.plot(
                x, y,
                label=modelo,
                color=cores_flip[modelo],
                marker=marcadores_flip[modelo],
                linewidth=2,
                markersize=6
            )

            # Plotagem do intervalo de confiança (95%)
            plt.fill_between(
                x, y_low, y_high,
                color=cores_flip[modelo],
                alpha=0.15
            )

        plt.title(f"Estabilidade ao Ruído: Flip-Rate ({q} Quantis)", fontsize=15, fontweight="bold", pad=15)
        plt.xlabel("Sigma", fontsize=12, fontweight="bold")
        plt.ylabel("Flip-Rate Médio (%)", fontsize=12, fontweight="bold")

        plt.legend(fontsize=11)
        plt.tight_layout()

        # Salvando em PNG e PDF (alta resolução para documentação)
        plt.savefig(os.path.join(out_dir, f"flip_rate_{q}Q.png"), dpi=600, bbox_inches="tight")
        plt.savefig(os.path.join(out_dir, f"flip_rate_{q}Q.pdf"), dpi=600, bbox_inches="tight")
        plt.close()

    # =========================================================
    # 3. GRÁFICOS DE DESEMPENHO (BARRAS)
    # =========================================================

    print("-> Gerando gráficos de desempenho...")

    df_perf = pd.read_csv(csv_performance, sep=";", decimal=",")

    # Limpeza e filtro dinâmico de famílias esperadas
    familias_esperadas = ["HERMES", "Baseline Aleatório"]
    familias_presentes = set(df_perf["familia"].unique())

    familias_ausentes = [f for f in familias_esperadas if f not in familias_presentes]
    if familias_ausentes:
        print("⚠️ Aviso: famílias não encontradas no CSV: " + ", ".join(familias_ausentes))

    df_perf = df_perf[df_perf["familia"].isin(familias_esperadas)].copy()
    quantis_perf = sorted(df_perf["q"].unique())

    # Dicionário de configuração individual das métricas
    metricas = {
        "acc_10": {"titulo": "Acurácia — Top 10%", "rotulo_y": "Acurácia Média (%)", "arquivo": "accuracy_10"},
        "acc_25": {"titulo": "Acurácia — Top 25%", "rotulo_y": "Acurácia Média (%)", "arquivo": "accuracy_25"},
        "acc_total": {"titulo": "Acurácia — Base Total", "rotulo_y": "Acurácia Média (%)", "arquivo": "accuracy_total"},
        "f1_10": {"titulo": "F1-score — Top 10%", "rotulo_y": "F1-score Médio (%)", "arquivo": "f1_10"},
        "f1_25": {"titulo": "F1-score — Top 25%", "rotulo_y": "F1-score Médio (%)", "arquivo": "f1_25"},
        "f1_total": {"titulo": "F1-score — Base Total", "rotulo_y": "F1-score Médio (%)", "arquivo": "f1_total"},
        "f1_macro_10": {"titulo": "F1 Macro — Top 10%", "rotulo_y": "F1 Macro Médio (%)", "arquivo": "f1_macro_10"},
        "f1_macro_25": {"titulo": "F1 Macro — Top 25%", "rotulo_y": "F1 Macro Médio (%)", "arquivo": "f1_macro_25"},
        "f1_macro_total": {"titulo": "F1 Macro — Base Total", "rotulo_y": "F1 Macro Médio (%)", "arquivo": "f1_macro_total"}
    }

    cores_familias = {
        "HERMES": "#1f77b4",
        "Baseline Aleatório": "#d62728"
    }

    for coluna_metrica, configuracao in metricas.items():
        if coluna_metrica not in df_perf.columns:
            continue

        for q in quantis_perf:
            df_q = df_perf[df_perf["q"] == q].copy()
            if df_q.empty:
                continue

            # Escalonando valores para %
            df_q["valor_percentual"] = df_q[coluna_metrica] * 100.0

            # Garantindo a ordem de plotagem na legenda
            modelos_h = df_q.loc[df_q["familia"] == "HERMES", "modelo"].unique().tolist()
            modelos_b = df_q.loc[df_q["familia"] == "Baseline Aleatório", "modelo"].unique().tolist()
            ordem_modelos = modelos_h + modelos_b

            if not ordem_modelos:
                continue

            # Mapeamento dinâmico de cores para abranger diferentes sigmas do mesmo modelo
            cores_modelos = {}
            for modelo in modelos_h: cores_modelos[modelo] = cores_familias["HERMES"]
            for modelo in modelos_b: cores_modelos[modelo] = cores_familias["Baseline Aleatório"]

            plt.figure(figsize=(16, 7))

            # CORREÇÃO DO WARNING SEABORN:
            # Ao utilizar o argumento `hue="modelo"` junto a `legend=False`,
            # mantemos o mapeamento de cores consistente através do argumento `palette`
            # sem gerar caixas de legenda redundantes na visualização final.
            sns.barplot(
                data=df_q,
                x="modelo",
                y="valor_percentual",
                hue="modelo",
                order=ordem_modelos,
                palette=cores_modelos,
                errorbar="sd",
                capsize=0.10,
                edgecolor="black",
                linewidth=0.9,
                legend=False
            )

            plt.title(f"Desempenho Comparativo: {configuracao['titulo']} ({q} Quantis)", fontsize=15, fontweight="bold", pad=15)
            plt.ylabel(configuracao["rotulo_y"], fontsize=12, fontweight="bold")
            plt.xlabel("Metodologia e Sigma", fontsize=12, fontweight="bold")

            # Rotacionando os labels para evitar sobreposição em listas longas
            plt.xticks(rotation=25, ha="right", fontsize=10)

            # Ajuste inteligente do eixo Y
            maximo = df_q["valor_percentual"].max()
            if pd.notna(maximo):
                plt.ylim(0, min(maximo * 1.25, 100))

            plt.tight_layout()

            # Exportação
            arquivo_png = os.path.join(out_dir, f"barras_{configuracao['arquivo']}_{q}Q.png")
            arquivo_pdf = os.path.join(out_dir, f"barras_{configuracao['arquivo']}_{q}Q.pdf")

            plt.savefig(arquivo_png, dpi=600, bbox_inches="tight")
            plt.savefig(arquivo_pdf, dpi=600, bbox_inches="tight")
            plt.close()

    print(f"-> Gráficos gerados com sucesso em: {out_dir}")

-> Lendo dados da pasta: D:\pycharm_projects\HERMES_diabetes_and_student\student_performance
-> Gerando gráficos de Flip-Rate...
-> Gerando gráficos de desempenho...
-> Gráficos gerados com sucesso em: D:\pycharm_projects\HERMES_diabetes_and_student\student_performance


In [4]:
# ============================================================
# GERADOR DE TABELA LATEX: TESTE DE WILCOXON ONE-SAMPLE
# ============================================================
import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

out_dir = os.path.join(os.getcwd(), "student_performance")
csv_perf = os.path.join(out_dir, "dados_sigmas_escolhidos.csv")
arquivo_saida = os.path.join(out_dir, "tabela_wilcoxon_student_performance.tex")

if os.path.exists(csv_perf):
    df_perf = pd.read_csv(csv_perf, sep=";", decimal=",")

    FAMILIA_HERMES = "HERMES"
    FAMILIA_B = "Baseline Aleatório"
    df_perf = df_perf[df_perf["familia"].isin([FAMILIA_HERMES, FAMILIA_B])]

    quantis = sorted(df_perf["q"].unique())
    resultados_wilcoxon = []

    def executar_wilcoxon_one_sample(valores_baseline, valor_hermes):
        diferencas = np.asarray(valores_baseline, dtype=float) - float(valor_hermes)
        diferencas = diferencas[np.isfinite(diferencas)]
        if len(diferencas) == 0: return np.nan, np.nan
        if np.allclose(diferencas, 0.0): return 0.0, 1.0
        try:
            return wilcoxon(diferencas, alternative="less", zero_method="wilcox")
        except ValueError:
            return np.nan, 1.0

    for q in quantis:
        df_q = df_perf[df_perf["q"] == q]
        df_h = df_q[df_q["familia"] == FAMILIA_HERMES]
        df_b = df_q[df_q["familia"] == FAMILIA_B]

        # Seleção pela maior métrica média
        melhor_h_acc = df_h.groupby("modelo")["acc_total"].mean().idxmax()
        melhor_b_acc = df_b.groupby("modelo")["acc_total"].mean().idxmax()

        acc_h = df_h.loc[df_h["modelo"] == melhor_h_acc, "acc_total"].iloc[0]
        acc_b = df_b[df_b["modelo"] == melhor_b_acc].sort_values("rodada")["acc_total"].to_numpy()

        _, p_acc = executar_wilcoxon_one_sample(acc_b, acc_h)

        # F1-Score
        melhor_h_f1 = df_h.groupby("modelo")["f1_total"].mean().idxmax()
        melhor_b_f1 = df_b.groupby("modelo")["f1_total"].mean().idxmax()

        f1_h = df_h.loc[df_h["modelo"] == melhor_h_f1, "f1_total"].iloc[0]
        f1_b = df_b[df_b["modelo"] == melhor_b_f1].sort_values("rodada")["f1_total"].to_numpy()

        _, p_f1 = executar_wilcoxon_one_sample(f1_b, f1_h)

        resultados_wilcoxon.append({
            "quantil": q, "melhor_h_acc": melhor_h_acc, "melhor_b_acc": melhor_b_acc, "p_value_acc": p_acc,
            "melhor_h_f1": melhor_h_f1, "melhor_b_f1": melhor_b_f1, "p_value_f1": p_f1
        })

    df_wilcoxon = pd.DataFrame(resultados_wilcoxon)

    # Geração LaTeX
    latex_str = [
        "% \\usepackage{booktabs} \\usepackage{graphicx}",
        "\\begin{table*}[htbp]",
        "  \\centering",
        "  \\caption{Teste de Wilcoxon one-sample: Baseline Aleatório vs HERMES.}",
        "  \\label{tab:wilcoxon_student_performance}",
        "  \\resizebox{\\textwidth}{!}{%",
        "  \\begin{tabular}{lcccc}",
        "    \\toprule",
        "    \\textbf{Quantil} & \\textbf{Melhor HERMES (Acurácia)} & \\textbf{$p$-value Baseline $<$ HERMES} & \\textbf{Melhor HERMES (F1)} & \\textbf{$p$-value Baseline $<$ HERMES} \\\\",
        "    \\midrule"
    ]

    for _, row in df_wilcoxon.iterrows():
        p_acc = row["p_value_acc"]
        p_f1 = row["p_value_f1"]
        texto_p_acc = f"\\textbf{{{p_acc:.4f}}}" if p_acc < 0.05 else f"{p_acc:.4f}"
        texto_p_f1 = f"\\textbf{{{p_f1:.4f}}}" if p_f1 < 0.05 else f"{p_f1:.4f}"

        latex_str.append(f"    {int(row['quantil'])} Quantis & {row['melhor_h_acc']} & {texto_p_acc} & {row['melhor_h_f1']} & {texto_p_f1} \\\\")

    latex_str.extend(["    \\bottomrule", "  \\end{tabular}%", "  }", "\\end{table*}"])

    with open(arquivo_saida, "w", encoding="utf-8") as arquivo:
        arquivo.write("\n".join(latex_str))

In [5]:
# ============================================================
# GERADOR DE TABELAS LATEX
# HERMES vs Baseline Aleatório
#
# Arquivo de entrada: diabetes_binary/dados_sigmas_escolhidos.csv
# Arquivos gerados:
# - tabela_accuracy_diabetes.tex
# - tabela_f1_diabetes.tex
# - tabela_f1_macro_diabetes.tex
# ============================================================

import os
import re
import pandas as pd
import warnings

# Ignorar avisos do pandas sobre encadeamento de atribuições se ocorrerem
warnings.filterwarnings("ignore")

# ============================================================
# 1. CAMINHOS E LEITURA DO CSV
# ============================================================
out_dir = os.path.join(os.getcwd(), "student_performance")
csv_path = os.path.join(out_dir, "dados_sigmas_escolhidos.csv")

if not os.path.exists(csv_path):
    print(f"⚠️ Arquivo não encontrado: {csv_path}")
    print("Execute primeiro a célula de avaliação de desempenho para gerar os resultados.")
else:
    df_tabela = pd.read_csv(csv_path, sep=";", decimal=",")

    # Filtragem atualizada para a nova metodologia
    familias_esperadas = ["HERMES", "Baseline Aleatório"]

    # Validação de integridade dos dados
    familias_presentes = set(df_tabela["familia"].unique())
    familias_ausentes = [f for f in familias_esperadas if f not in familias_presentes]

    if familias_ausentes:
        print("⚠️ Aviso: As seguintes metodologias não foram encontradas no CSV:")
        for familia in familias_ausentes:
            print(f"   - {familia}")

    # Mantém apenas os métodos pertinentes ao escopo atual
    df_tabela = df_tabela[df_tabela["familia"].isin(familias_esperadas)].copy()

    if df_tabela.empty:
        raise ValueError("Erro: Nenhum registro de HERMES ou Baseline Aleatório encontrado para tabular.")

    print("✅ Dados encontrados com sucesso! Gerando tabelas estruturadas em LaTeX...\n")

    # ========================================================
    # 2. FUNÇÕES AUXILIARES DE FORMATAÇÃO E ORDENAÇÃO
    # ========================================================

    def extrair_sigma(nome_modelo: str) -> float:
        """Extrai o valor numérico de dispersão (\sigma) usando Expressão Regular."""
        resultado = re.search(r"(\d+\.\d+)", str(nome_modelo))
        return float(resultado.group(1)) if resultado else 0.0

    def obter_chave_ordenacao(linha: pd.Series) -> tuple:
        """Garante a hierarquia de apresentação: HERMES primeiro, Baseline depois; ordenados por sigma."""
        familia = linha["familia"]
        sigma = extrair_sigma(linha["modelo"])
        ordem_familia = {"HERMES": 0, "Baseline Aleatório": 1}
        return (ordem_familia.get(familia, 99), sigma)

    def formatar_celula(media: float, desvio: float, familia: str) -> str:
        """
        Formata o texto da célula com base na natureza do modelo.
        HERMES é determinístico na fase de difusão (exibe apenas a média).
        Baseline é estocástico (exibe média \pm desvio padrão).
        """
        if familia == "HERMES" or pd.isna(desvio) or desvio == 0.0:
            return f"{media:.2f}"
        return f"{media:.2f} $\\pm$ {desvio:.2f}"

    def gerar_tabela_latex(
        dataframe: pd.DataFrame,
        colunas_metricas: tuple,
        titulo_tabela: str,
        label_tabela: str,
        arquivo_saida: str,
        titulo_10: str,
        titulo_25: str,
        titulo_total: str
    ):
        """Função core que consolida as estatísticas e escreve o código LaTeX nativo."""
        col_10, col_25, col_total = colunas_metricas
        df = dataframe.copy()

        # Conversão de proporção (0.0 - 1.0) para percentual (0% - 100%)
        for coluna in [col_10, col_25, col_total]:
            df[coluna] = df[coluna] * 100.0

        # Agregação estatística vetorial via pandas
        df_stats = (
            df.groupby(["q", "modelo", "familia"])[[col_10, col_25, col_total]]
            .agg(["mean", "std"])
            .reset_index()
        )

        # Achatamento do MultiIndex para colunas planas
        df_stats.columns = [
            "q", "modelo", "familia",
            "met_10_mean", "met_10_std",
            "met_25_mean", "met_25_std",
            "met_total_mean", "met_total_std"
        ]

        # Substitui possíveis NaNs (gerados pela ausência de variância no HERMES) por 0.0
        df_stats = df_stats.fillna(0.0)

        # Aplicação da formatação textual
        df_stats["Top 10\\%"] = df_stats.apply(lambda row: formatar_celula(row["met_10_mean"], row["met_10_std"], row["familia"]), axis=1)
        df_stats["Top 25\\%"] = df_stats.apply(lambda row: formatar_celula(row["met_25_mean"], row["met_25_std"], row["familia"]), axis=1)
        df_stats["Total"] = df_stats.apply(lambda row: formatar_celula(row["met_total_mean"], row["met_total_std"], row["familia"]), axis=1)

        # Aplicação da ordenação personalizada
        df_stats["ordem_chave"] = df_stats.apply(obter_chave_ordenacao, axis=1)
        df_stats = df_stats.sort_values(["q", "ordem_chave"]).drop(columns=["ordem_chave"])

        # ====================================================
        # Construção da Estrutura LaTeX (Tabela)
        # ====================================================
        latex = [
            "% Requerimentos do preâmbulo LaTeX: \\usepackage{booktabs} \\usepackage{graphicx}",
            "\\begin{table*}[htbp]",
            "  \\centering",
            f"  \\caption{{{titulo_tabela}}}",
            f"  \\label{{{label_tabela}}}",
            "  \\resizebox{\\textwidth}{!}{%",
            "  \\begin{tabular}{llccc}",
            "    \\toprule",
            f"    \\textbf{{Quantil}} & \\textbf{{Metodologia}} & \\textbf{{{titulo_10} (\\%)}} & \\textbf{{{titulo_25} (\\%)}} & \\textbf{{{titulo_total} (\\%)}} \\\\",
            "    \\midrule"
        ]

        ultimo_q = None
        for _, linha in df_stats.iterrows():
            q_atual = linha["q"]

            # Adiciona um divisor visual (\midrule) entre os blocos de quantis
            if ultimo_q is not None and q_atual != ultimo_q:
                latex.append("    \\midrule")

            texto_quantil = f"{q_atual} Quantis" if q_atual != ultimo_q else ""
            modelo = str(linha["modelo"])

            latex.append(
                f"    {texto_quantil} & {modelo} & "
                f"{linha['Top 10\\%']} & {linha['Top 25\\%']} & {linha['Total']} \\\\"
            )
            ultimo_q = q_atual

        latex.extend([
            "    \\bottomrule",
            "  \\end{tabular}%",
            "  }",
            "\\end{table*}\n"
        ])

        codigo_latex = "\n".join(latex)
        caminho_saida = os.path.join(out_dir, arquivo_saida)

        # Escrita no disco com codificação segura
        with open(caminho_saida, "w", encoding="utf-8") as arquivo:
            arquivo.write(codigo_latex)

        print(f"-> Arquivo LaTeX gerado com sucesso: {arquivo_saida}")
        return df_stats

    # ========================================================
    # 3. GERAÇÃO DAS TRÊS TABELAS DE DESEMPENHO
    # ========================================================

    # Tabela 1: Acurácia
    gerar_tabela_latex(
        dataframe=df_tabela,
        colunas_metricas=("acc_10", "acc_25", "acc_total"),
        titulo_tabela="Desempenho Comparativo de Acurácia: Média $\\pm$ Desvio Padrão (\\%)",
        label_tabela="tab:accuracy_student_performance",
        arquivo_saida="tabela_accuracy_student_performance.tex",
        titulo_10="Acurácia Top 10\\%",
        titulo_25="Acurácia Top 25\\%",
        titulo_total="Acurácia Total"
    )

    # Tabela 2: F1-Score (Classe Positiva)
    gerar_tabela_latex(
        dataframe=df_tabela,
        colunas_metricas=("f1_10", "f1_25", "f1_total"),
        titulo_tabela="Desempenho Comparativo de F1-score da Classe Positiva: Média $\\pm$ Desvio Padrão (\\%)",
        label_tabela="tab:f1_student_performance",
        arquivo_saida="tabela_f1_student_performance.tex",
        titulo_10="F1 Top 10\\%",
        titulo_25="F1 Top 25\\%",
        titulo_total="F1 Total"
    )

    # Tabela 3: F1-Score Macro
    gerar_tabela_latex(
        dataframe=df_tabela,
        colunas_metricas=("f1_macro_10", "f1_macro_25", "f1_macro_total"),
        titulo_tabela="Desempenho Comparativo de F1 Macro: Média $\\pm$ Desvio Padrão (\\%)",
        label_tabela="tab:f1_macro_student_performance",
        arquivo_saida="tabela_f1_macro_student_performance.tex",
        titulo_10="F1 Macro Top 10\\%",
        titulo_25="F1 Macro Top 25\\%",
        titulo_total="F1 Macro Total"
    )

✅ Dados encontrados com sucesso! Gerando tabelas estruturadas em LaTeX...

-> Arquivo LaTeX gerado com sucesso: tabela_accuracy_student_performance.tex
-> Arquivo LaTeX gerado com sucesso: tabela_f1_student_performance.tex
-> Arquivo LaTeX gerado com sucesso: tabela_f1_macro_student_performance.tex


# Verificação de balanceamento

In [7]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


def verificar_distribuicao_student_performance():
    print(
        "-> Baixando base Student Performance do OpenML "
        "(data_id=42352) para auditoria..."
    )

    dados = fetch_openml(data_id=42352, as_frame=True)
    df = dados.frame.copy()

    df["G3"] = pd.to_numeric(df["G3"], errors="coerce")
    df = df.dropna(subset=["G3"]).reset_index(drop=True)

    df["target"] = (df["G3"] >= 10).astype(int)

    contagem = df["target"].value_counts().sort_index()
    proporcao = df["target"].value_counts(normalize=True).sort_index() * 100

    print("\n--- Auditoria da Base Student Performance ---")
    print(
        f"Classe 0 (G3 < 10): "
        f"{contagem.get(0, 0)} amostras "
        f"({proporcao.get(0, 0):.2f}%)"
    )

    print(
        f"Classe 1 (G3 >= 10): "
        f"{contagem.get(1, 0)} amostras "
        f"({proporcao.get(1, 0):.2f}%)"
    )

    diferenca = abs(proporcao.get(0, 0) - proporcao.get(1, 0))

    print(f"\n-> Diferença entre classes: {diferenca:.2f} p.p.")

    if diferenca < 5:
        print("-> Resultado: a base apresenta distribuição aproximadamente equilibrada.")
    else:
        print("-> Alerta: a base apresenta desbalanceamento entre as classes.")


verificar_distribuicao_student_performance()

-> Baixando base Student Performance do OpenML (data_id=42352) para auditoria...

--- Auditoria da Base Student Performance ---
Classe 0 (G3 < 10): 130 amostras (32.91%)
Classe 1 (G3 >= 10): 265 amostras (67.09%)

-> Diferença entre classes: 34.18 p.p.
-> Alerta: a base apresenta desbalanceamento entre as classes.
